# Learning art by names or by looking: all results

This notebook reproduces every result in the write-up *"Learning art by names or by looking: how the training signal reshapes a vision model's view of art"*, for both models and both learning signals, with 3 seeds each.

| Condition | Model | Learning signal | Training |
|---|---|---|---|
| 1 | DINOv2-small | labels (predict style) | last 4 blocks, learning rate 3e-5 |
| 2 | DINOv2-small | exposure (self-supervised, no names) | last 4 blocks, 3e-5 |
| 3 | DINOv3-small | labels | last 4 blocks, 3e-5 |
| 4 | DINOv3-small | exposure | last 8 blocks, 1e-4 (so it moves as far as DINOv3 label training) |

All conditions: 960 WikiArt paintings (8 styles × 120), 30 epochs, 3-epoch head warm-up, gradient clipping, 3 seeds.

**Part A trains and saves; Parts B, C and D analyse.**
- **Part A** saves every finished run to `RESULTS_DIR` and skips runs already saved. The whole thing takes roughly 80–90 minutes on a T4. You can split it across sessions: set `RESULTS_DIR` to a Google Drive folder (cell below) so results survive a disconnect, and rerun; it continues where it stopped.
- **Parts B, C and D** need only the saved results (plus the paintings, already loaded at the top), so they run on any runtime, even without a GPU: Part B in a few minutes, Parts C and D in about 20–25 minutes more (most of it tuning the MLP probe).

**Results and the write-up**

| Part B section | Write-up |
|---|---|
| 1. Style accessibility vs organisation | Table 1 |
| 2. Vector or field | Table 2 |
| 3. Local patch structure | Section 3 |
| 4. Abstract vs representational, four measures | Table 4 (+ the DINOv3 rise-and-fall, now with painting-level measures) |
| 5. Specificity: style pairs, controls, alternative groupings | Table 5 |
| 6. Embedding maps | Figure |
| 7. Verdicts | Discussion |
| **Part C** · controls and null tests: test paintings only, centroid decomposition, general spreading, exact nulls, image statistics | Robustness section |
| **Part D** · style probes: accessibility vs presence, test-set intervals, leakage-free few-shot probes | Section 4.1 |

Supplementary analyses from earlier notebooks (attention maps, the warm-up dip test, the symmetry toy test) are not repeated here.

**Before running:** T4 GPU for Part A; your `HF_TOKEN` secret for DINOv3.

In [ ]:
!pip -q install datasets umap-learn "transformers>=4.56"

In [ ]:
import os
os.environ["USE_TF"] = "0"

import copy, gc, math, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms as T
from datasets import load_dataset
from transformers import AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import umap

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU attached. Switch to a T4 GPU runtime.")
import json
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

**Optional: keep results across sessions.** Uncomment to save to Google Drive (Colab), then use the Drive path as `RESULTS_DIR` below. **Kaggle:** uncomment the token lines instead (needed for DINOv3).

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")          # then set RESULTS_DIR = "/content/drive/MyDrive/art_results"

# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Settings

In [ ]:
RUNS = [
    dict(model="facebook/dinov2-small", objective="labels", lr=3e-5, n_blocks=4),
    dict(model="facebook/dinov2-small", objective="exposure", lr=3e-5, n_blocks=4),
    dict(model="facebook/dinov3-vits16-pretrain-lvd1689m", objective="labels", lr=3e-5, n_blocks=4),
    dict(model="facebook/dinov3-vits16-pretrain-lvd1689m", objective="exposure", lr=1e-4, n_blocks=8),
]
MODELS = list(dict.fromkeys(r["model"] for r in RUNS))
SEEDS = [0, 1, 2]
RESULTS_DIR = "results"         # a Google Drive path keeps results across sessions
TRAIN = True                    # False: skip Part A and analyse whatever is saved

EPOCHS = 30
WARMUP = 3                      # head warm-up epochs (model frozen)
RECORD_EVERY, DENSE_UNTIL = 2, 6
BATCH = {"labels": 32, "exposure": 64}
SSL_TEMPERATURE = 0.1
USE_AMP = True                  # mixed precision (DINOv2)
AMP_OFF_FOR = ["dinov3"]        # DINOv3 needs full precision (16-bit overflows → NaN)
GRAD_CLIP = 1.0
K_NEIGHBOURS = 10
TH = {"centroid between": 0.05, "pairwise ratio": 0.05, "held-out d′": 0.05, "neighbour mixing": 0.02}

STYLES = ["Impressionism", "Baroque", "Cubism", "Ukiyo_e", "Pop_Art",
          "Abstract_Expressionism", "Realism", "Color_Field_Painting"]
PER_STYLE, MAX_SCAN = 120, 25000
ABSTRACT = ["Abstract_Expressionism", "Color_Field_Painting"]
REPRESENTATIONAL = ["Baroque", "Realism", "Impressionism", "Ukiyo_e"]
QUERY = (0.35, 0.5)

RECORD_EPOCHS = sorted(set([0] + list(range(1, min(DENSE_UNTIL, EPOCHS) + 1))
                           + list(range(RECORD_EVERY, EPOCHS + 1, RECORD_EVERY)) + [EPOCHS]))
print("Recording at epochs:", RECORD_EPOCHS)

## Load paintings

Same 960 paintings and splits as every earlier notebook (fixed seed).

In [ ]:
torch.manual_seed(0); np.random.seed(0); random.seed(0)
ds = load_dataset("huggan/wikiart", split="train", streaming=True)
if ds.features is None:
    ds = ds._resolve_features()
all_styles = ds.features["style"].names
style_ids = {all_styles.index(s): s for s in STYLES if s in all_styles}
id_to_label = {sid: i for i, sid in enumerate(style_ids)}
class_names = [style_ids[sid] for sid in style_ids]
K = len(class_names)

counts = {sid: 0 for sid in style_ids}
images, labels = [], []
for i, ex in enumerate(ds.shuffle(seed=0, buffer_size=1000)):
    sid = ex["style"]
    if sid in counts and counts[sid] < PER_STYLE:
        img = ex["image"].convert("RGB")
        img.thumbnail((320, 320))
        images.append(img); labels.append(id_to_label[sid]); counts[sid] += 1
    if all(c >= PER_STYLE for c in counts.values()) or i >= MAX_SCAN:
        break
assert all(c == PER_STYLE for c in counts.values()), f"only found {counts} paintings per style; raise MAX_SCAN"
labels = np.array(labels)
del ds; gc.collect()

tr_idx, te_idx = train_test_split(np.arange(len(images)), test_size=0.2, stratify=labels, random_state=0)
fit_idx, val_idx = train_test_split(tr_idx, test_size=0.2, stratify=labels[tr_idx], random_state=0)
ia_all = np.where(np.isin(labels, [class_names.index(n) for n in ABSTRACT if n in class_names]))[0]
ir_all = np.where(np.isin(labels, [class_names.index(n) for n in REPRESENTATIONAL if n in class_names]))[0]
test_imgs = [images[j] for j in te_idx]
EX_POS = [int(np.where(labels[te_idx] == k)[0][0]) for k in range(min(3, K))]   # example paintings for the maps
print(f"Kept {len(images)} paintings: {len(fit_idx)} fit / {len(val_idx)} validation / {len(te_idx)} test")

## Helpers

In [ ]:
def short(model_id):
    return model_id.split("/")[-1].replace("-pretrain-lvd1689m", "")

def probe_on(emb, train_ix, eval_ix):
    sc = StandardScaler().fit(emb[train_ix])
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(emb[train_ix]), labels[train_ix])
    return clf.score(sc.transform(emb[eval_ix]), labels[eval_ix])

probe = lambda emb: probe_on(emb, tr_idx, te_idx)
val_probe = lambda emb: probe_on(emb, fit_idx, val_idx)

def split_parts(emb, ia, ir):
    """between: cosine distance between the abstract and representational centres;
       within: average cosine distance of paintings to their own group's centre;
       overall: average cosine distance of all paintings to the overall centre (general spread)."""
    e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    A, R_ = e[ia], e[ir]
    ca, cr = A.mean(0), R_.mean(0)
    between = 1 - ca @ cr / (np.linalg.norm(ca) * np.linalg.norm(cr))
    within = np.mean([np.mean(1 - X @ c / np.linalg.norm(c)) for X, c in ((A, ca), (R_, cr))])
    g = e.mean(0)
    overall = np.mean(1 - e @ g / np.linalg.norm(g))
    return between, within, overall

def split_ratio(emb, ia, ir):
    between, within, _ = split_parts(emb, ia, ir)
    return between / within

def split_record(emb):
    b, w, o = split_parts(emb, ia_all, ir_all)
    return dict(split_between=b, split_within=w, overall_spread=o)

def linear_cka(X, Y):
    X = X.astype(np.float64) - X.mean(0); Y = Y.astype(np.float64) - Y.mean(0)
    return np.linalg.norm(Y.T @ X) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))

def show_img(img):
    """Same framing the models see: shorter side to 256, centre crop 224."""
    w, h = img.size
    sc = 256 / min(w, h)
    img = img.resize((max(256, round(w * sc)), max(256, round(h * sc))), Image.BICUBIC)
    w, h = img.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return img.crop((l, t, l + 224, t + 224))

def upsample(a, size=224):
    a = np.asarray(Image.fromarray(a.astype(np.float32)).resize((size, size), Image.BILINEAR))
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def unit_rows(E):
    E = E.astype(np.float64)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def procrustes_rotation(B, A):
    """Rotation R (orthogonal) that best maps B onto A, fitted on centred data."""
    U, _, Vt = np.linalg.svd((B - B.mean(0)).T @ (A - A.mean(0)))
    return U @ Vt

def field_stats(E0, Et, align=True):
    """Split how paintings moved (E0 -> Et, unit-normalised) into:
       shared   = one common shift for all paintings (the steering *vector*),
       style    = extra movement shared within each style (a style-organised *field*),
       other    = movement specific to individual paintings.
    Fractions of total squared movement; they add to 1."""
    A, B = unit_rows(E0), unit_rows(Et)
    if align:
        B = B @ procrustes_rotation(B, A)       # remove any pure rotation of the space
    D = B - A
    total = (D ** 2).sum(1).mean()
    mean_shift = D.mean(0)
    shared = (mean_shift ** 2).sum()
    Dr = D - mean_shift
    style = sum((labels == k).sum() * (Dr[labels == k].mean(0) ** 2).sum() for k in range(K)) / len(labels)
    u = A[ia_all].mean(0) - A[ir_all].mean(0); u /= np.linalg.norm(u)
    proj = Dr @ u                                # + = towards the abstract side of the original axis
    return dict(total=total, shared=shared / total, style=style / total, other=1 - (shared + style) / total,
                abstract_move=proj[ia_all].mean(), representational_move=proj[ir_all].mean(), D=D, Dr=Dr, B=B, A=A)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
label_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
ssl_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.3, 1.0)), T.RandomHorizontalFlip(),
                    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
                    T.ToTensor(), T.Normalize(MEAN, STD)])
AMP = USE_AMP and device == "cuda"

def load_backbone(model_id):
    return AutoModel.from_pretrained(model_id, attn_implementation="eager").to(device).eval()

def cls_of(model, x):
    return model(pixel_values=x).last_hidden_state[:, 0]

@torch.no_grad()
def embed(model, imgs, bs=64):
    model.eval()
    return torch.cat([cls_of(model, torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)).float().cpu()
                      for i in range(0, len(imgs), bs)]).numpy()

def get_blocks(bb):
    for path in ("encoder.layer", "model.layer", "layer"):
        obj = bb
        try:
            for name in path.split("."):
                obj = getattr(obj, name)
        except AttributeError:
            continue
        if isinstance(obj, nn.ModuleList):
            return obj
    raise ValueError("Could not find the transformer blocks")

def get_final_norm(bb):
    return bb.layernorm if hasattr(bb, "layernorm") else bb.norm

def make_head(objective, hidden):
    if objective == "labels":
        return nn.Linear(hidden, K)
    return nn.Sequential(nn.Linear(hidden, 512), nn.GELU(), nn.Linear(512, 128))

def nt_xent(z, tau):
    z = F.normalize(z.float(), dim=1)            # computed in float32 for stability
    n = z.shape[0] // 2
    sim = z @ z.T / tau - 1e9 * torch.eye(2 * n, device=z.device)
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

@torch.no_grad()
def head_accuracy(model, head, index, bs=64):
    model.eval(); head.eval()
    correct = 0
    for i in range(0, len(index), bs):
        b = index[i:i + bs]
        x = torch.stack([eval_tf(images[j]) for j in b]).to(device)
        correct += int((head(cls_of(model, x)).argmax(1).cpu().numpy() == labels[b]).sum())
    return correct / len(index)

@torch.no_grad()
def trainable_change(base, model):
    pb = dict(base.named_parameters())
    d = n = 0.0
    for name, p in model.named_parameters():
        if p.requires_grad:
            d += (p - pb[name]).float().pow(2).sum().item(); n += pb[name].float().pow(2).sum().item()
    return math.sqrt(d) / (math.sqrt(n) + 1e-12)

@torch.no_grad()
def patch_sims(model, imgs, bs=32):
    """Per painting: cosine similarity between every pair of final-layer patch tokens. (images, P, P), float16 on GPU."""
    model.eval()
    out = []
    for i in range(0, len(imgs), bs):
        px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
        h = model(pixel_values=px).last_hidden_state[:, 1:]
        side = int(math.sqrt(h.shape[1]))
        h = F.normalize(h[:, -side * side:].float(), dim=-1)     # patch tokens only (drops register tokens)
        out.append((h @ h.transpose(1, 2)).half())
    return torch.cat(out)

_MASKS = {}
def grid_masks(side, dev):
    if side not in _MASKS:
        yy, xx = torch.meshgrid(torch.arange(side), torch.arange(side), indexing="ij")
        c = torch.stack([yy.flatten(), xx.flatten()], 1).float()
        d = torch.cdist(c, c)
        near = (d > 0) & (d <= 1.5)                                # the 8 neighbouring patches
        far = d >= side / 2                                        # patches at least half the image apart
        triu = torch.triu(torch.ones_like(d, dtype=torch.bool), diagonal=1)
        _MASKS[side] = (near.to(dev), far.to(dev), triu.to(dev))
    return _MASKS[side]

@torch.no_grad()
def spatial_metrics(S, S0):
    """local: neighbours minus distant similarity (higher = more spatially organised);
       homog: mean similarity of all patch pairs (higher = patches more alike, global info swamping local);
       preserve: correlation of each painting's patch-similarity pattern with the untrained model's (1 = unchanged)."""
    near, far, triu = grid_masks(int(math.sqrt(S.shape[-1])), S.device)
    S, S0 = S.float(), S0.float()
    local = S[:, near].mean(1) - S[:, far].mean(1)
    a, b = S[:, triu], S0[:, triu]
    homog = a.mean(1)
    a = a - a.mean(1, keepdim=True); b = b - b.mean(1, keepdim=True)
    preserve = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)
    return dict(spatial_local=local.mean().item(), spatial_homog=homog.mean().item(),
                spatial_preserve=preserve.mean().item())

def train_long(base, objective, seed, on_record, lr, n_blocks, warmup=0):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = copy.deepcopy(base).to(device)
    for p in model.parameters():
        p.requires_grad = False
    blocks = get_blocks(model)
    for block in blocks[max(len(blocks) - n_blocks, 0):]:
        for p in block.parameters():
            p.requires_grad = True
    for p in get_final_norm(model).parameters():
        p.requires_grad = True
    head = make_head(objective, model.config.hidden_size).to(device)
    bs = BATCH[objective]

    # Head warm-up: train only the new head on the frozen model, so random head weights can't disturb the features
    head_opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.05)
    for _ in range(warmup):
        model.eval(); head.train()
        order = np.random.permutation(fit_idx)
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                feats = cls_of(model, x).float()
            if objective == "labels":
                loss = F.cross_entropy(head(feats), torch.tensor(labels[b]).to(device))
            else:
                loss = nt_xent(head(feats), SSL_TEMPERATURE)
            head_opt.zero_grad(set_to_none=True); loss.backward(); head_opt.step()
    del head_opt

    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if p.requires_grad], "lr": lr},
        {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    trainable = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
    for epoch in range(1, EPOCHS + 1):
        model.train(); head.train()
        order, losses = np.random.permutation(fit_idx), []
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            if len(b) < 2:
                continue
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                if objective == "labels":
                    x = torch.stack([label_tf(images[j]) for j in b]).to(device)
                    loss = F.cross_entropy(head(cls_of(model, x)).float(), torch.tensor(labels[b]).to(device))
                else:
                    x = torch.stack([ssl_tf(images[j]) for j in b] + [ssl_tf(images[j]) for j in b]).to(device)
                    loss = nt_xent(head(cls_of(model, x)), SSL_TEMPERATURE)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if GRAD_CLIP:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        if epoch in RECORD_EPOCHS:
            on_record(epoch, model, head, float(np.mean(losses)))
    del model, head, opt

# Part A: train and save

For each model: the untrained embeddings and patch structure are saved once; then each condition and seed is trained and saved (embeddings of all 960 paintings at every recorded epoch, plus final patch-structure measures). Runs already saved are skipped.

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
fname = lambda model_id, tag: os.path.join(RESULTS_DIR, f"{short(model_id)}__{tag}")
t0 = time.time()
clock = lambda: f"[{(time.time() - t0) / 60:.1f} min]"

for model_id in MODELS:
    todo = [(r, s) for r in RUNS if r["model"] == model_id for s in SEEDS
            if not os.path.exists(fname(model_id, f"{r['objective']}_s{s}.npz"))]
    before_done = os.path.exists(fname(model_id, "before.npz"))
    if not TRAIN or (not todo and before_done):
        print(f"{short(model_id)}: nothing to train" + ("" if TRAIN else " (TRAIN = False)"))
        continue
    AMP = USE_AMP and device == "cuda" and not any(k in model_id for k in AMP_OFF_FOR)
    print(f"\n=== {short(model_id)} (mixed precision: {AMP}); {len(todo)} runs to do ===")
    base = load_backbone(model_id)
    S0 = patch_sims(base, test_imgs)
    if not before_done:
        np.savez_compressed(fname(model_id, "before.npz"), emb=embed(base, images).astype(np.float16), labels=labels)
        json.dump(spatial_metrics(S0, S0), open(fname(model_id, "before_spatial.json"), "w"))
    for r, seed in todo:
        embs, final = {}, {}
        def on_record(ep, m, h, loss):
            if not np.isfinite(loss):
                raise RuntimeError(f"{short(model_id)} {r['objective']} seed {seed}: loss is {loss} at epoch {ep}")
            embs[ep] = embed(m, images).astype(np.float16)
            if ep == EPOCHS:
                S = patch_sims(m, test_imgs)
                final.update(spatial_metrics(S, S0), weight_change=trainable_change(base, m))
                del S
            if ep % 10 == 0:
                print(f"  {r['objective']} seed {seed}, epoch {ep}: loss {loss:.3f} {clock()}")
        train_long(base, r["objective"], seed, on_record, lr=r["lr"], n_blocks=r["n_blocks"], warmup=WARMUP)
        np.savez_compressed(fname(model_id, f"{r['objective']}_s{seed}.npz"),
                            labels=labels, **{f"ep{e}": v for e, v in embs.items()})
        json.dump(dict(final, lr=r["lr"], n_blocks=r["n_blocks"]), open(fname(model_id, f"{r['objective']}_s{seed}.json"), "w"))
        print(f"  saved {r['objective']} seed {seed} {clock()}")
        gc.collect(); torch.cuda.empty_cache()
    del base, S0; gc.collect(); torch.cuda.empty_cache()
print(f"\nPart A done {clock()}")

# Part B: analysis

Everything below uses only the saved results.

In [ ]:
fname = lambda model_id, tag: os.path.join(RESULTS_DIR, f"{short(model_id)}__{tag}")
BEFORE, RES = {}, {}
for model_id in MODELS:
    p = fname(model_id, "before.npz")
    if not os.path.exists(p):
        print(f"{short(model_id)}: no saved results"); continue
    d = np.load(p)
    assert np.array_equal(d["labels"], labels), "Saved results were made with different paintings"
    BEFORE[model_id] = dict(emb=d["emb"].astype(np.float32),
                            spatial=json.load(open(fname(model_id, "before_spatial.json"))))
    for r in [r for r in RUNS if r["model"] == model_id]:
        for s in SEEDS:
            p = fname(model_id, f"{r['objective']}_s{s}.npz")
            if not os.path.exists(p):
                continue
            d = np.load(p)
            embs = {int(k[2:]): d[k].astype(np.float32) for k in d.files if k.startswith("ep")}
            embs[0] = BEFORE[model_id]["emb"]
            RES[(model_id, r["objective"], s)] = dict(embs=embs, final=json.load(open(fname(model_id, f"{r['objective']}_s{s}.json"))))

CONDS = list(dict.fromkeys((m, o) for (m, o, s) in RES))
NAME = {c: f"{short(c[0])} · {c[1]}" for c in CONDS}
SEEDS_OF = {c: sorted(s for (m, o, s) in RES if (m, o) == c) for c in CONDS}
EPS = sorted(set.intersection(*[set(v["embs"]) for v in RES.values()])) if RES else []
COL = {c: plt.get_cmap("tab10")(i) for i, c in enumerate(CONDS)}
LS = {c: ("--" if c[1] == "exposure" else "-") for c in CONDS}
STYLE_CMAP = plt.get_cmap("tab10")
emb_at = lambda c, s, ep: RES[(c[0], c[1], s)]["embs"][ep]
last = EPS[-1] if EPS else None
ms = lambda v: f"{np.mean(v):.3f} ± {np.std(v, ddof=1) if len(v) > 1 else 0:.3f}"
for c in CONDS:
    print(f"{NAME[c]}: seeds {SEEDS_OF[c]}")
if not CONDS:
    raise RuntimeError("No saved results found; run Part A first.")

## 1. Style accessibility vs organisation

Style probe = how readily style can be read out by a linear boundary (accessibility; Part D adds k-NN, a tuned MLP and test-set intervals); silhouette = how tightly the 8 styles cluster; CKA = similarity to the untrained model (1 = unchanged). Mean ± sd over seeds at epoch 30.

In [ ]:
rows = []
for c in CONDS:
    e0 = BEFORE[c[0]]["emb"]
    pr = [probe(emb_at(c, s, last)) for s in SEEDS_OF[c]]
    si = [silhouette_score(emb_at(c, s, last), labels, metric="cosine") for s in SEEDS_OF[c]]
    ck = [linear_cka(e0, emb_at(c, s, last)) for s in SEEDS_OF[c]]
    rows.append({"condition": NAME[c], "probe before": f"{probe(e0):.3f}", "probe after": ms(pr),
                 "silhouette before": f"{silhouette_score(e0, labels, metric='cosine'):.3f}",
                 "silhouette after": ms(si), "CKA to untrained": ms(ck)})
print(pd.DataFrame(rows).set_index("condition").to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for c in CONDS:
    for ax, f in ((axes[0], lambda e: silhouette_score(e, labels, metric="cosine")),
                  (axes[1], lambda e: linear_cka(BEFORE[c[0]]["emb"], e))):
        mat = np.array([[f(emb_at(c, s, ep)) for ep in EPS] for s in SEEDS_OF[c]])
        for line in mat:
            ax.plot(EPS, line, color=COL[c], ls=LS[c], alpha=0.3, lw=1)
        ax.plot(EPS, mat.mean(0), color=COL[c], ls=LS[c], lw=2.5, label=NAME[c])
axes[0].set_title("Style cluster tightness (silhouette)"); axes[1].set_title("Similarity to the untrained model (CKA)")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Vector or field

Each painting's movement from the untrained model (unit-normalised, rotation removed), split into a **shared** shift, a **style-organised** part and an **individual** part. Mean over seeds. Below: whether labels and exposure move paintings in related directions (seed-matched cosine of the non-shared movement).

In [ ]:
FIELD = {}
for c in CONDS:
    vals = [field_stats(BEFORE[c[0]]["emb"], emb_at(c, s, last)) for s in SEEDS_OF[c]]
    FIELD[c] = {k: [v[k] for v in vals] for k in ("shared", "style", "other")}
    FIELD[c]["Dr"] = [v["Dr"] for v in vals]
print(pd.DataFrame([{"condition": NAME[c], "shared": ms(FIELD[c]["shared"]), "style-organised": ms(FIELD[c]["style"]),
                     "individual": ms(FIELD[c]["other"])} for c in CONDS]).set_index("condition").to_string())

fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(CONDS)); left = np.zeros(len(CONDS))
for part, colour, label in (("shared", "#9ecae1", "shared (vector)"), ("style", "#fd8d3c", "style-organised (field)"),
                            ("other", "#d9d9d9", "individual")):
    v = np.array([np.mean(FIELD[c][part]) for c in CONDS])
    ax.barh(y, v, left=left, color=colour, label=label); left += v
ax.set_yticks(y); ax.set_yticklabels([NAME[c] for c in CONDS]); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel("fraction of total movement"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

for m in MODELS:
    cl, ce = (m, "labels"), (m, "exposure")
    if cl in FIELD and ce in FIELD:
        cos = []
        for Dl, De in zip(FIELD[cl]["Dr"], FIELD[ce]["Dr"]):
            cos.append(np.mean((Dl * De).sum(1) / (np.linalg.norm(Dl, axis=1) * np.linalg.norm(De, axis=1) + 1e-12)))
        print(f"{short(m)}: labels vs exposure movement, mean cosine {ms(cos)}")

## 3. Local patch structure

Measured within each of the 192 test paintings from final-layer patch tokens. **Patch sameness** = average similarity between patches (up = patches more alike); **local contrast** = neighbouring minus distant patch similarity; **structure kept** = correlation of each painting's patch-similarity pattern with the untrained model's (1 = unchanged).

In [ ]:
rows = []
for c in CONDS:
    b = BEFORE[c[0]]["spatial"]
    f = [RES[(c[0], c[1], s)]["final"] for s in SEEDS_OF[c]]
    rows.append({"condition": NAME[c],
                 "patch sameness": f"{b['spatial_homog']:.3f} → " + ms([x["spatial_homog"] for x in f]),
                 "local contrast": f"{b['spatial_local']:.3f} → " + ms([x["spatial_local"] for x in f]),
                 "structure kept": ms([x["spatial_preserve"] for x in f])})
print(pd.DataFrame(rows).set_index("condition").to_string())

## 4. Abstract vs representational: four measures

- **centroid distance:** between the average abstract and average representational painting;
- **pairwise ratio:** average distance between cross-group pairs of paintings ÷ within-group pairs;
- **neighbour mixing:** share of each painting's 10 nearest neighbours from the other group, relative to chance (reported as absolute change);
- **held-out d′:** separation along a linear boundary learned on other paintings.

For neighbour mixing *up* = closer; for the others *down* = closer. Curves: change from before training (thin = seeds, thick = mean; dashed = exposure).

In [ ]:
def group_idx(names):
    return np.where(np.isin(labels, [class_names.index(n) for n in names if n in class_names]))[0]

def centroid_between(emb, ga, gb):
    e = unit_rows(emb); ca, cb = e[ga].mean(0), e[gb].mean(0)
    return 1 - ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb))

def pairwise_ratio(emb, ga, gb):
    e = unit_rows(emb); A, B = e[ga], e[gb]
    def within(X):
        n = len(X); return 1 - ((X @ X.T).sum() - n) / (n * (n - 1))
    return (1 - (A @ B.T).mean()) / ((within(A) + within(B)) / 2)

def knn_mixing(emb, ga, gb, k=K_NEIGHBOURS):
    idx = np.concatenate([ga, gb]); grp = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    e = unit_rows(emb[idx]); S = e @ e.T
    np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    other = (grp[nn] != grp[:, None]).mean(1)
    return float((other / (np.where(grp == 0, len(gb), len(ga)) / (len(idx) - 1))).mean())

def heldout_dprime(emb, ga, gb, folds=5):
    X = unit_rows(emb[np.r_[ga, gb]]); y = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    vals = []
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=0).split(X, y):
        p = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto").fit(X[tr], y[tr]).transform(X[te])[:, 0]
        a, b = p[y[te] == 0], p[y[te] == 1]
        vals.append(abs(a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2))
    return float(np.mean(vals))

MEASURES = {"centroid between": centroid_between, "pairwise ratio": pairwise_ratio,
            "neighbour mixing": knn_mixing, "held-out d′": heldout_dprime}
CLOSER_IF = {"centroid between": "down", "pairwise ratio": "down", "neighbour mixing": "up", "held-out d′": "down"}
ABSOLUTE = {"neighbour mixing"}
change = lambda m, new, old: new - old if m in ABSOLUTE else new / old - 1
is_closer = lambda m, c: c < -TH[m] if CLOSER_IF[m] == "down" else c > TH[m]
is_apart = lambda m, c: c > TH[m] if CLOSER_IF[m] == "down" else c < -TH[m]
GA, GB = group_idx(ABSTRACT), group_idx(REPRESENTATIONAL)

TRACK = {(c, s, ep): {m: f(emb_at(c, s, ep), GA, GB) for m, f in MEASURES.items()}
         for c in CONDS for s in SEEDS_OF[c] for ep in EPS}

fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, m in zip(axes, MEASURES):
    for c in CONDS:
        mat = np.array([[change(m, TRACK[(c, s, ep)][m], TRACK[(c, s, 0)][m]) for ep in EPS] for s in SEEDS_OF[c]])
        for line in mat:
            ax.plot(EPS, line, color=COL[c], ls=LS[c], alpha=0.3, lw=1)
        ax.plot(EPS, mat.mean(0), color=COL[c], ls=LS[c], lw=2.5, label=NAME[c])
    ax.axhline(0, color="k", ls=":", lw=1)
    ax.set_title(f"{m} ({'absolute' if m in ABSOLUTE else 'relative'} change)\ncloser = {CLOSER_IF[m]}", fontsize=10)
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

FINAL = pd.DataFrame([dict(condition=NAME[c], seed=s, **{m: change(m, TRACK[(c, s, last)][m], TRACK[(c, s, 0)][m]) for m in MEASURES})
                      for c in CONDS for s in SEEDS_OF[c]])
rows = []
for c in CONDS:
    g = FINAL[FINAL.condition == NAME[c]]; n = len(g)
    row = {"condition": NAME[c]}
    for m in MEASURES:
        cl = sum(is_closer(m, v) for v in g[m]); ap = sum(is_apart(m, v) for v in g[m])
        row[m] = f"{ms(list(g[m]))} (closer {cl}/{n}, apart {ap}/{n})"
    rows.append(row)
print("Change from before training to the final epoch (relative; neighbour mixing absolute):")
print(pd.DataFrame(rows).set_index("condition").to_string())

print("\nPeak of the pairwise ratio during training (labels):")
for c in [c for c in CONDS if c[1] == "labels"]:
    mean_curve = np.mean([[TRACK[(c, s, ep)]["pairwise ratio"] for ep in EPS] for s in SEEDS_OF[c]], axis=0)
    i = int(np.argmax(mean_curve))
    print(f"  {NAME[c]}: peak at epoch {EPS[i]} ({mean_curve[i] / mean_curve[0] - 1:+.1%}), "
          f"final {mean_curve[-1] / mean_curve[0] - 1:+.1%}")

## 5. Specificity: every style pair, controls and alternative groupings

**Style-pair map:** change in the pairwise ratio for every pair of the 8 styles (blue = closer). Pairs crossing the abstract–representational divide are outlined. **Controls** compare groups on the same side of the divide; **alternative groupings** redraw the line between abstract and representational. These groupings and controls are our own design choices, fixed before the robustness runs.

In [ ]:
Kc = len(class_names)
A_SET, R_SET = set(ABSTRACT), set(REPRESENTATIONAL)
crosses = lambda i, j: bool({class_names[i], class_names[j]} & A_SET) and bool({class_names[i], class_names[j]} & R_SET)
IU = [(i, j) for i in range(Kc) for j in range(i + 1, Kc)]
PAIR = {}
for c in CONDS:
    M = np.zeros((Kc, Kc))
    for s in SEEDS_OF[c]:
        e0, e1 = emb_at(c, s, 0), emb_at(c, s, last)
        for i, j in IU:
            gi, gj = np.where(labels == i)[0], np.where(labels == j)[0]
            M[i, j] += change("pairwise ratio", pairwise_ratio(e1, gi, gj), pairwise_ratio(e0, gi, gj)) / len(SEEDS_OF[c])
            M[j, i] = M[i, j]
    PAIR[c] = M

v = max(np.abs(M).max() for M in PAIR.values())
ncol = 2; nrow = int(np.ceil(len(CONDS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 6 * nrow), squeeze=False, constrained_layout=True)
for ax in axes.ravel()[len(CONDS):]:
    ax.axis("off")
for ax, c in zip(axes.ravel(), CONDS):
    M = PAIR[c]
    im = ax.imshow(np.where(np.eye(Kc, dtype=bool), np.nan, M), cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_xticks(range(Kc)); ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(Kc)); ax.set_yticklabels(class_names, fontsize=8)
    for i in range(Kc):
        for j in range(Kc):
            if i != j:
                ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=7)
                if crosses(i, j):
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.5))
    ax.set_title(NAME[c], fontsize=10)
fig.colorbar(im, ax=axes.ravel().tolist(), label="change in pairwise ratio (negative = closer)", shrink=0.6)
plt.show()

GROUPINGS = {
    "original": (ABSTRACT, REPRESENTATIONAL),
    "Cubism counted as abstract": (ABSTRACT + ["Cubism"], REPRESENTATIONAL),
    "Pop Art counted as representational": (ABSTRACT, REPRESENTATIONAL + ["Pop_Art"]),
    "Abstract Expressionism only": (["Abstract_Expressionism"], REPRESENTATIONAL),
    "Colour Field only": (["Color_Field_Painting"], REPRESENTATIONAL),
    "without Ukiyo-e": (ABSTRACT, [s for s in REPRESENTATIONAL if s != "Ukiyo_e"]),
    "without Impressionism": (ABSTRACT, [s for s in REPRESENTATIONAL if s != "Impressionism"]),
    "European figurative only (Baroque, Realism)": (ABSTRACT, ["Baroque", "Realism"]),
    "control: Baroque + Realism vs Impressionism + Ukiyo-e": (["Baroque", "Realism"], ["Impressionism", "Ukiyo_e"]),
    "control: Cubism vs Pop Art": (["Cubism"], ["Pop_Art"]),
    "control: Abstract Expressionism vs Colour Field": (["Abstract_Expressionism"], ["Color_Field_Painting"]),
}
GTAB = []
for gname, (a, b) in GROUPINGS.items():
    ga, gb = group_idx(a), group_idx(b)
    for c in CONDS:
        pr, km = [], []
        for s in SEEDS_OF[c]:
            e0, e1 = emb_at(c, s, 0), emb_at(c, s, last)
            pr.append(change("pairwise ratio", pairwise_ratio(e1, ga, gb), pairwise_ratio(e0, ga, gb)))
            km.append(change("neighbour mixing", knn_mixing(e1, ga, gb), knn_mixing(e0, ga, gb)))
        both = sum(is_closer("pairwise ratio", p) and is_closer("neighbour mixing", k) for p, k in zip(pr, km))
        GTAB.append(dict(grouping=gname, condition=NAME[c], pairwise_ratio=np.mean(pr), neighbour_mixing=np.mean(km),
                         closer_seeds=f"{both}/{len(pr)}"))
GTAB = pd.DataFrame(GTAB)

rows = []
for c in CONDS:
    M = PAIR[c]
    t = GTAB[GTAB.condition == NAME[c]]
    ctrl = t[t.grouping.str.startswith("control")]
    alt = t[~t.grouping.str.startswith("control")]
    rows.append({"condition": NAME[c],
                 "crossing pairs (n=8)": f"{np.mean([M[i, j] for i, j in IU if crosses(i, j)]):+.3f}",
                 "other pairs (n=20)": f"{np.mean([M[i, j] for i, j in IU if not crosses(i, j)]):+.3f}",
                 "controls closer": f"{sum(is_closer('pairwise ratio', x) for x in ctrl.pairwise_ratio)}/{len(ctrl)}",
                 "control changes": ", ".join(f"{x:+.3f}" for x in ctrl.pairwise_ratio),
                 "alternative groupings closer": f"{sum(is_closer('pairwise ratio', x) for x in alt.pairwise_ratio)}/{len(alt)}"})
print(pd.DataFrame(rows).set_index("condition").to_string())
for c in CONDS:
    print(f"\n{NAME[c]}")
    print(GTAB[GTAB.condition == NAME[c]].drop(columns="condition").set_index("grouping").round(3).to_string())

## 6. Embedding maps

UMAP of all 960 paintings (seed 0), coloured by style: one row per model, untrained then each learning signal. Read neighbourhoods, not distances between clusters.

In [ ]:
umap_kw = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0)
rows_ = [[("before", m)] + [(c, None) for c in CONDS if c[0] == m] for m in BEFORE]
ncol = max(len(r) for r in rows_)
fig, axes = plt.subplots(len(rows_), ncol, figsize=(5 * ncol, 4.6 * len(rows_)), squeeze=False)
for i, row in enumerate(rows_):
    for j in range(ncol):
        ax = axes[i, j]
        if j >= len(row):
            ax.axis("off"); continue
        key, m = row[j]
        if key == "before":
            e, title = BEFORE[m]["emb"], f"{short(m)} · untrained"
        else:
            s0 = SEEDS_OF[key][0]
            e, title = emb_at(key, s0, last), f"{NAME[key]} (seed {s0})"
        xy = umap.UMAP(**umap_kw).fit_transform(e)
        for k, cname in enumerate(class_names):
            msk = labels == k
            ax.scatter(xy[msk, 0], xy[msk, 1], s=5, color=STYLE_CMAP(k), label=cname, alpha=0.8)
        ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="center right", markerscale=3, fontsize=9)
plt.tight_layout(rect=(0, 0, 0.86, 1)); plt.show()

## 7. Verdicts

Two summaries per condition: the **pre-set strict rule** (all four measures agree in every seed) and the **painting-level measures** alone (pairwise ratio, neighbour mixing, held-out d′), which compare individual paintings rather than group averages. Both are reported; the write-up states the strict result and explains the difference.

In [ ]:
for c in CONDS:
    g = FINAL[FINAL.condition == NAME[c]]; n = len(g)
    t = GTAB[GTAB.condition == NAME[c]]
    ctrl = t[t.grouping.str.startswith("control")]; alt = t[~t.grouping.str.startswith("control")]
    M = PAIR[c]
    cross = np.mean([M[i, j] for i, j in IU if crosses(i, j)]); other = np.mean([M[i, j] for i, j in IU if not crosses(i, j)])
    def informative(m):
        """Neighbour mixing can't register change if the groups never share neighbours (stuck near 0)."""
        if m not in ABSOLUTE:
            return True
        return max(TRACK[(c, s, ep)][m] for s in SEEDS_OF[c] for ep in (0, last)) > TH[m]
    def agree(measures, test):
        used = [m for m in measures if informative(m)]
        return sum(all(test(m, g.iloc[k][m]) for m in used) for k in range(n)) if used else 0
    painting = ["pairwise ratio", "neighbour mixing", "held-out d′"]
    skipped = [m for m in MEASURES if not informative(m)]
    print(f"\n{NAME[c]}")
    if skipped:
        print(f"  (left out of the counts, stuck near zero: {skipped})")
    print(f"  Strict rule (all 4 measures): closer in {agree(MEASURES, is_closer)}/{n} seeds, apart in {agree(MEASURES, is_apart)}/{n}")
    print(f"  Painting-level measures (3):  closer in {agree(painting, is_closer)}/{n} seeds, apart in {agree(painting, is_apart)}/{n}")
    print(f"  Pairwise ratio + neighbour mixing: closer in {agree(painting[:2], is_closer)}/{n}, apart in {agree(painting[:2], is_apart)}/{n}")
    print(f"  Controls closer: {sum(is_closer('pairwise ratio', x) for x in ctrl.pairwise_ratio)}/{len(ctrl)}; "
          f"alternative groupings closer: {sum(is_closer('pairwise ratio', x) for x in alt.pairwise_ratio)}/{len(alt)}")
    th = TH["pairwise ratio"]
    if cross < 0 and cross < other - th:
        spec = "crossing pairs converge more than others → specific to the abstraction divide"
    elif cross > 0 and cross > other + th:
        spec = "crossing pairs separate more than others → extra separation across the abstraction divide"
    elif abs(cross - other) <= th:
        spec = "crossing and other pairs change alike → not specific to the abstraction divide"
    else:
        spec = "crossing pairs change less than other pairs → not specific to the abstraction divide"
    print(f"  Crossing pairs {cross:+.3f} vs other pairs {other:+.3f}: {spec}")

# Part C: controls and null tests

Everything here reads the saved embeddings and the paintings loaded at the top; nothing is trained.

| # | Check | Question it answers |
|---|---|---|
| 1 | Test paintings only | Do the results hold on the 192 paintings no model was trained on? |
| 2 | Centroid decomposition | Why does the centroid measure disagree with the painting-level measures? |
| 3 | General spreading | Could the pairwise-ratio convergence be a side effect of all paintings spreading apart? |
| 4 | Exact nulls | Is the crossing-pair convergence beyond chance across all 420 possible groupings? Is the style-organised share above chance? |
| 5 | Image statistics | Does the crossing-pair convergence survive controlling for simple image statistics? |

In [ ]:
import itertools
from scipy import ndimage
from scipy.stats import binomtest

# Parts C and D keep their own names for models, conditions and embeddings (E0, E1, CONDS_S ...).
# Part C re-defines the measure functions with identical definitions, so Part B's numbers do not change;
# still, run the notebook top to bottom rather than re-running Part B cells after Part C.
LAST_EP = last
N_FIELD_NULL = 200
MODELS_S = [short(m) for m in MODELS if m in BEFORE]
E0 = {short(m): BEFORE[m]["emb"] for m in BEFORE}
E1 = {(short(m), o, s): v["embs"][LAST_EP] for (m, o, s), v in RES.items()}
CONDS_S = [(m, o) for m in MODELS_S for o in ["labels", "exposure"] if any((m, o, s) in E1 for s in SEEDS)]
SEEDS_OF_S = {c: [s for s in SEEDS if (c[0], c[1], s) in E1] for c in CONDS_S}
NAME_S = {c: f"{c[0]} · {c[1]}" for c in CONDS_S}
for c in CONDS_S:
    print(f"{NAME_S[c]}: seeds {SEEDS_OF_S[c]}")

## C0. Measures used in Part C

In [ ]:
def unit_rows(E):
    E = E.astype(np.float64)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def idx_of(names, pool=None):
    ix = np.where(np.isin(labels, [class_names.index(n) for n in names]))[0]
    return ix if pool is None else np.intersect1d(ix, pool)

def centroid_between(emb, ga, gb):
    e = unit_rows(emb); ca, cb = e[ga].mean(0), e[gb].mean(0)
    return 1 - ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb))

def pairwise_ratio(emb, ga, gb):
    e = unit_rows(emb); A, B = e[ga], e[gb]
    def within(X):
        n = len(X); return 1 - ((X @ X.T).sum() - n) / (n * (n - 1))
    return (1 - (A @ B.T).mean()) / ((within(A) + within(B)) / 2)

def knn_mixing(emb, ga, gb, k=K_NEIGHBOURS):
    idx = np.concatenate([ga, gb]); grp = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    e = unit_rows(emb[idx]); S = e @ e.T
    np.fill_diagonal(S, -np.inf)
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    other = (grp[nn] != grp[:, None]).mean(1)
    return float((other / (np.where(grp == 0, len(gb), len(ga)) / (len(idx) - 1))).mean())

def heldout_dprime(emb, ga, gb, folds=5):
    X = unit_rows(emb[np.r_[ga, gb]]); y = np.r_[np.zeros(len(ga)), np.ones(len(gb))]
    vals = []
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=0).split(X, y):
        p = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto").fit(X[tr], y[tr]).transform(X[te])[:, 0]
        a, b = p[y[te] == 0], p[y[te] == 1]
        vals.append(abs(a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2))
    return float(np.mean(vals))

MEASURES = {"centroid between": centroid_between, "pairwise ratio": pairwise_ratio,
            "neighbour mixing": knn_mixing, "held-out d′": heldout_dprime}
CLOSER_IF = {"centroid between": "down", "pairwise ratio": "down", "neighbour mixing": "up", "held-out d′": "down"}
change = lambda m, new, old: new - old if m == "neighbour mixing" else new / old - 1
is_closer = lambda m, c: c < -TH[m] if CLOSER_IF[m] == "down" else c > TH[m]
is_apart = lambda m, c: c > TH[m] if CLOSER_IF[m] == "down" else c < -TH[m]
ms_s = lambda v: f"{np.mean(v):+.3f} ± {np.std(v, ddof=1) if len(v) > 1 else 0:.3f}"

A_SET, R_SET = set(ABSTRACT), set(REPRESENTATIONAL)
IU = [(i, j) for i in range(K) for j in range(i + 1, K)]
def crossing_mask(a_names, r_names):
    a, r = set(a_names), set(r_names)
    return np.array([(class_names[i] in a and class_names[j] in r) or (class_names[i] in r and class_names[j] in a)
                     for i, j in IU])
CROSS = crossing_mask(ABSTRACT, REPRESENTATIONAL)

def pair_changes(e0, e1, pool=None, measure="pairwise ratio"):
    """Change in a measure for each of the 28 style pairs (optionally within a subset of paintings)."""
    out = []
    for i, j in IU:
        gi, gj = idx_of([class_names[i]], pool), idx_of([class_names[j]], pool)
        f = MEASURES[measure]
        out.append(change(measure, f(e1, gi, gj), f(e0, gi, gj)))
    return np.array(out)

SUMMARY_C = {}

## C1. test paintings only

All earlier measures used all 960 paintings, 614 of which each model trained on. Here every measure is computed on the **192 test paintings** only (48 abstract, 96 representational), before vs epoch 30.

In [ ]:
GA_T, GB_T = idx_of(ABSTRACT, te_idx), idx_of(REPRESENTATIONAL, te_idx)
print(f"test set: {len(GA_T)} abstract, {len(GB_T)} representational\n")
rows, PAIR_TEST = [], {}
for c in CONDS_S:
    e0 = E0[c[0]]
    vals = {m: [] for m in MEASURES}
    pairs = []
    for s in SEEDS_OF_S[c]:
        e1 = E1[(c[0], c[1], s)]
        for m, f in MEASURES.items():
            vals[m].append(change(m, f(e1, GA_T, GB_T), f(e0, GA_T, GB_T)))
        pairs.append(pair_changes(e0, e1, pool=te_idx))
    PAIR_TEST[c] = np.mean(pairs, 0)
    row = {"condition": NAME_S[c]}
    for m in MEASURES:
        n = len(vals[m])
        row[m] = (f"{ms_s(vals[m])} (closer {sum(is_closer(m, v) for v in vals[m])}/{n}, "
                  f"apart {sum(is_apart(m, v) for v in vals[m])}/{n})")
    row["crossing pairs"] = f"{PAIR_TEST[c][CROSS].mean():+.3f}"
    row["other pairs"] = f"{PAIR_TEST[c][~CROSS].mean():+.3f}"
    rows.append(row)
    SUMMARY_C[f"test-only · {NAME_S[c]}"] = {m: round(float(np.mean(v)), 4) for m, v in vals.items()} | {
        "crossing pairs": round(float(PAIR_TEST[c][CROSS].mean()), 4),
        "other pairs": round(float(PAIR_TEST[c][~CROSS].mean()), 4)}
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).set_index("condition").to_string())

## C2. why the centroid measure disagrees

For unit vectors, every measure here is built from the same three numbers: the mean similarity **across** groups, and the mean similarity **within** each group.

- Centroid cosine = mean cross similarity ÷ (‖abstract centroid‖ × ‖representational centroid‖), where ‖centroid‖² = mean within-group similarity (including each painting with itself).
- Pairwise ratio = (1 − mean cross similarity) ÷ (1 − mean within similarity), i.e. a ratio of **distances**.

So the centroid measure is a ratio of *similarities* and the pairwise ratio a ratio of *distances*. When all similarities shrink (paintings spread out), the two can move in opposite directions. The table shows each ingredient before and after.

In [ ]:
GA, GB = idx_of(ABSTRACT), idx_of(REPRESENTATIONAL)
def ingredients(emb, ga=GA, gb=GB):
    e = unit_rows(emb); A, B = e[ga], e[gb]
    ca, cb = A.mean(0), B.mean(0)
    off = lambda X: ((X @ X.T).sum() - len(X)) / (len(X) * (len(X) - 1))
    return dict(cross_sim=float((A @ B.T).mean()), norm_A=float(np.linalg.norm(ca)), norm_B=float(np.linalg.norm(cb)),
                within_sim_A=float(off(A)), within_sim_B=float(off(B)),
                centroid_cos=float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb))))

rows = []
for m in MODELS_S:
    r = ingredients(E0[m]); r["condition"] = f"{m} · before"; rows.append(r)
for c in CONDS_S:
    vals = [ingredients(E1[(c[0], c[1], s)]) for s in SEEDS_OF_S[c]]
    r = {k: float(np.mean([v[k] for v in vals])) for k in vals[0]}; r["condition"] = NAME_S[c] + " · after"; rows.append(r)
ING = pd.DataFrame(rows).set_index("condition")
print("Mean of 3 seeds after training. A = abstract, B = representational.\n")
print(ING.round(4).to_string())

print("\nWhich ingredient moves the centroid distance? Relative change in centroid distance if only one part had changed:")
print("  only_cross_sim: after-training cross similarity, before-training centroid norms")
print("  only_norms:     before-training cross similarity, after-training centroid norms")
print("  (positive = apart). Compare with the pairwise ratio, built from the same similarities as distances.\n")
cdist = lambda cross, na, nb: 1 - cross / (na * nb)
pratio = lambda cross, wa, wb: (1 - cross) / ((1 - wa + 1 - wb) / 2)
rows = []
for c in CONDS_S:
    b, a = ING.loc[f"{c[0]} · before"], ING.loc[NAME_S[c] + " · after"]
    d0 = cdist(b.cross_sim, b.norm_A, b.norm_B)
    t = dict(condition=NAME_S[c],
             centroid_distance=cdist(a.cross_sim, a.norm_A, a.norm_B) / d0 - 1,
             only_cross_sim=cdist(a.cross_sim, b.norm_A, b.norm_B) / d0 - 1,
             only_norms=cdist(b.cross_sim, a.norm_A, a.norm_B) / d0 - 1,
             norm_A_change=a.norm_A / b.norm_A - 1, norm_B_change=a.norm_B / b.norm_B - 1,
             cross_sim_change=a.cross_sim - b.cross_sim,
             within_sim_A_change=a.within_sim_A - b.within_sim_A, within_sim_B_change=a.within_sim_B - b.within_sim_B,
             pairwise_ratio=pratio(a.cross_sim, a.within_sim_A, a.within_sim_B) / pratio(b.cross_sim, b.within_sim_A, b.within_sim_B) - 1)
    rows.append(t)
    SUMMARY_C[f"centroid decomposition · {NAME_S[c]}"] = {k: round(float(v), 4) for k, v in t.items() if k != "condition"}
print(pd.DataFrame(rows).set_index("condition").round(4).T.to_string())

## C3. is the convergence just general spreading?

Contrastive training spreads all paintings apart (uniformity). If every similarity shrank by the same factor, every pairwise ratio would drift towards 1 — and pairs that start further apart would drift more. That would make crossing pairs look like they "converge" more without anything specific to abstraction.

Two tests:
1. **Spread-matched prediction:** shrink every similarity by the global factor observed for that condition, compute the predicted change for each style pair, and see whether crossing pairs converge **beyond** that prediction.
2. **Regression across the 28 style pairs:** change ~ crossing + starting log-ratio. If the crossing effect is only about starting distance, its coefficient goes to zero.

Neighbour mixing is rank-based, so a uniform shrink cannot change it; the per-pair neighbour-mixing change is also shown.

In [ ]:
def mean_offdiag_sim(emb):
    e = unit_rows(emb); n = len(e)
    s = e.sum(0)
    return float((s @ s - n) / (n * (n - 1)))

def pair_ratios(emb):
    return np.array([pairwise_ratio(emb, idx_of([class_names[i]]), idx_of([class_names[j]])) for i, j in IU])

def predicted_ratio_after_shrink(emb, k):
    """Pairwise ratio per style pair if every off-diagonal similarity were multiplied by k."""
    e = unit_rows(emb); out = []
    for i, j in IU:
        A, B = e[labels == i], e[labels == j]
        off = lambda X: ((X @ X.T).sum() - len(X)) / (len(X) * (len(X) - 1))
        cross = (A @ B.T).mean()
        out.append((1 - k * cross) / ((1 - k * off(A) + 1 - k * off(B)) / 2))
    return np.array(out)

def ols(y, X):
    X1 = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    return beta

PAIR_ALL, PAIR_KNN, SPREAD_ROWS = {}, {}, []
for c in CONDS_S:
    e0 = E0[c[0]]
    r0 = pair_ratios(e0)
    obs, pred, knn, ks = [], [], [], []
    for s in SEEDS_OF_S[c]:
        e1 = E1[(c[0], c[1], s)]
        k = mean_offdiag_sim(e1) / mean_offdiag_sim(e0); ks.append(k)
        obs.append(pair_ratios(e1) / r0 - 1)
        pred.append(predicted_ratio_after_shrink(e0, k) / r0 - 1)
        knn.append(pair_changes(e0, e1, measure="neighbour mixing"))
    obs, pred, knn = np.mean(obs, 0), np.mean(pred, 0), np.mean(knn, 0)
    PAIR_ALL[c], PAIR_KNN[c] = obs, knn
    excess = obs - pred
    beta = ols(obs, np.column_stack([CROSS.astype(float), np.log(r0)]))
    beta0 = ols(obs, CROSS.astype(float))
    row = dict(condition=NAME_S[c],
               mean_similarity_factor=np.mean(ks),
               crossing_observed=obs[CROSS].mean(), other_observed=obs[~CROSS].mean(),
               crossing_predicted_by_spread=pred[CROSS].mean(), other_predicted_by_spread=pred[~CROSS].mean(),
               crossing_excess=excess[CROSS].mean(), other_excess=excess[~CROSS].mean(),
               crossing_coef_alone=beta0[1], crossing_coef_with_start_distance=beta[1],
               knn_crossing=knn[CROSS].mean(), knn_other=knn[~CROSS].mean())
    SPREAD_ROWS.append(row)
    SUMMARY_C[f"spreading · {NAME_S[c]}"] = {k: round(float(v), 4) for k, v in row.items() if k != "condition"}
print("mean_similarity_factor < 1 means paintings spread apart overall.")
print("Pairwise-ratio changes are relative; neighbour-mixing (knn) changes are absolute.\n")
print(pd.DataFrame(SPREAD_ROWS).set_index("condition").round(4).T.to_string())

fig, axes = plt.subplots(1, len(CONDS_S), figsize=(4.2 * len(CONDS_S), 3.8), squeeze=False)
r0s = {m: pair_ratios(E0[m]) for m in MODELS_S}
for ax, c in zip(axes[0], CONDS_S):
    x = np.log(r0s[c[0]])
    ax.scatter(x[~CROSS], PAIR_ALL[c][~CROSS], c="grey", label="other pairs")
    ax.scatter(x[CROSS], PAIR_ALL[c][CROSS], c="crimson", label="crossing pairs")
    ax.axhline(0, color="k", lw=0.8, ls=":")
    ax.set_xlabel("log pairwise ratio before training"); ax.set_title(NAME_S[c], fontsize=9)
axes[0][0].set_ylabel("relative change in pairwise ratio"); axes[0][0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## C4. exact nulls

**Crossing vs other pairs.** With 8 styles there are exactly 420 ways to call 2 styles "abstract" and 4 "representational". For each, we compute the gap (mean change of its 8 crossing pairs − mean change of the other 20) and ask where the real grouping falls. This is an exact test, and it respects the fact that the 28 pairs share styles.

**Style-organised share.** The field decomposition gives a non-zero style share even for meaningless groups, because group means of finite samples are never exactly zero. We recompute the share with paintings randomly reassigned to 8 groups of 120.

In [ ]:
ALL_GROUPINGS = [(list(a), [s for s in r]) for a in itertools.combinations(class_names, 2)
                 for r in itertools.combinations([s for s in class_names if s not in a], 4)]
assert len(ALL_GROUPINGS) == 420
GROUPING_MASKS = np.array([crossing_mask(a, r) for a, r in ALL_GROUPINGS])

def exact_rank(values, mask_obs):
    gaps = np.array([values[m].mean() - values[~m].mean() for m in GROUPING_MASKS])
    obs = values[mask_obs].mean() - values[~mask_obs].mean()
    return obs, gaps, float((gaps <= obs + 1e-12).mean()), float((gaps >= obs - 1e-12).mean())

rows = []
for c in CONDS_S:
    for label, vals in (("pairwise ratio, all paintings", PAIR_ALL[c]), ("pairwise ratio, test paintings", PAIR_TEST[c]),
                        ("neighbour mixing, all paintings", PAIR_KNN[c])):
        obs, gaps, p_low, p_high = exact_rank(vals, CROSS)
        p = p_low if obs < 0 else p_high                     # one-sided, in the direction observed
        is_conv = (obs < 0) if "ratio" in label else (obs > 0)  # ratio down / mixing up = crossing pairs converge more
        rows.append(dict(condition=NAME_S[c], measure=label, observed_gap=obs,
                         direction="crossing pairs converge more" if is_conv else "crossing pairs separate more",
                         null_median=np.median(gaps), p_one_sided=p))
        SUMMARY_C[f"exact null · {NAME_S[c]} · {label}"] = dict(gap=round(float(obs), 4), p=round(p, 4))
print("p_one_sided: share of all 420 groupings with a gap at least as extreme, in the observed direction")
print("(1/420 = 0.0024 is the smallest possible value; the real grouping is included in the 420)\n")
print(pd.DataFrame(rows).set_index(["condition", "measure"]).round(4).to_string())

In [ ]:
def style_share(E0, E1, lab):
    A, B = unit_rows(E0), unit_rows(E1)
    U, _, Vt = np.linalg.svd((B - B.mean(0)).T @ (A - A.mean(0)))
    B = B @ (U @ Vt)
    D = B - A
    total = (D ** 2).sum(1).mean()
    Dr = D - D.mean(0)
    return sum((lab == k).sum() * (Dr[lab == k].mean(0) ** 2).sum() for k in np.unique(lab)) / len(lab) / total

rng = np.random.default_rng(0)
rows = []
for c in CONDS_S:
    e0 = E0[c[0]]
    obs = [style_share(e0, E1[(c[0], c[1], s)], labels) for s in SEEDS_OF_S[c]]
    e1 = E1[(c[0], c[1], SEEDS_OF_S[c][0])]
    null = np.array([style_share(e0, e1, rng.permutation(labels)) for _ in range(N_FIELD_NULL)])
    rows.append(dict(condition=NAME_S[c], observed_style_share=np.mean(obs), null_mean=null.mean(),
                     null_95th=np.quantile(null, 0.95), observed_over_null_mean=np.mean(obs) / null.mean()))
    SUMMARY_C[f"field null · {NAME_S[c]}"] = dict(observed=round(float(np.mean(obs)), 4), null_mean=round(float(null.mean()), 4),
                                              null_95th=round(float(np.quantile(null, 0.95)), 4))
print(f"Style-organised share vs random 8-way partitions ({N_FIELD_NULL} partitions, seed-0 embeddings):\n")
print(pd.DataFrame(rows).set_index("condition").round(4).to_string())

## C5. does the effect survive simple image statistics?

Uses the paintings loaded at the top. For each painting, on the same 224-pixel centre crop the models see:

- **mean luminance** and **RMS contrast** (luminance standard deviation)
- **saturation** and **colourfulness** (Hasler & Süsstrunk, 2003)
- **edge density** (share of pixels with a strong Sobel gradient)
- **flat-region share** (share of 8×8 blocks with almost no luminance variation)
- **spectral slope** (slope of the log power spectrum against log spatial frequency)
- a **colour histogram** (RGB, 8×8×8 bins)

Two analyses:
1. **Across the 28 style pairs:** regress the change in pairwise ratio on *crossing* + *starting distance* + *low-level distance* between the two styles (one statistic at a time, then a combined low-level distance). If the crossing coefficient stays, the effect is not explained by that statistic.
2. **Across painting pairs:** for every pair of abstract/representational paintings, regress the change in similarity on *crossing*, *same style*, *starting similarity*, and low-level distances (scalar statistics and colour histogram). Compare the crossing coefficient with and without the low-level terms.

In [ ]:
def centre_crop(img):
    w, h = img.size; sc = 256 / min(w, h)
    img = img.resize((max(256, round(w * sc)), max(256, round(h * sc))), Image.BICUBIC)
    w, h = img.size; l, t = (w - 224) // 2, (h - 224) // 2
    return np.asarray(img.crop((l, t, l + 224, t + 224)), dtype=np.float64) / 255.0

N = 224
yy, xx = np.indices((N, N)) - N // 2
RADIUS = np.hypot(yy, xx).astype(int)
HANN = np.outer(np.hanning(N), np.hanning(N))
FREQS = np.arange(4, 100)

def image_stats(img):
    x = centre_crop(img)
    R, G, B = x[..., 0], x[..., 1], x[..., 2]
    Y = 0.2126 * R + 0.7152 * G + 0.0722 * B
    mx, mn = x.max(2), x.min(2)
    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-8), 0)
    rg, yb = R - G, 0.5 * (R + G) - B
    colourful = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
    gm = np.hypot(ndimage.sobel(Y, 0), ndimage.sobel(Y, 1))
    blocks = Y.reshape(N // 8, 8, N // 8, 8).std(axis=(1, 3))
    P = np.abs(np.fft.fftshift(np.fft.fft2((Y - Y.mean()) * HANN))) ** 2
    radial = np.bincount(RADIUS.ravel(), P.ravel()) / np.bincount(RADIUS.ravel())
    slope = np.polyfit(np.log(FREQS), np.log(radial[FREQS] + 1e-12), 1)[0]
    hist = np.histogramdd(x.reshape(-1, 3), bins=(8, 8, 8), range=((0, 1),) * 3)[0].ravel()
    return dict(luminance=Y.mean(), contrast=Y.std(), saturation=sat.mean(), colourfulness=colourful,
                edge_density=(gm > 0.5).mean(), flat_share=(blocks < 0.02).mean(), spectral_slope=slope), hist / hist.sum()

stats, hists = zip(*[image_stats(im) for im in images])
STATS = pd.DataFrame(stats); HIST = np.array(hists)
STAT_NAMES = list(STATS.columns)
print("Style means of the image statistics:\n")
print(STATS.groupby(np.array(class_names)[labels]).mean().round(3).to_string())

In [ ]:
STAT_NAMES = [n for n in STATS.columns if STATS[n].std() > 1e-9]   # drop any statistic that never varies
Z = ((STATS[STAT_NAMES] - STATS[STAT_NAMES].mean()) / STATS[STAT_NAMES].std()).to_numpy()  # standardised per painting
SQH = np.sqrt(HIST)                                             # Hellinger embedding of the colour histogram

def style_mean(v, k):
    return v[labels == k].mean(0)

def js_distance(p, q):
    m = (p + q) / 2
    kl = lambda a, b: np.sum(np.where(a > 0, a * np.log(a / np.maximum(b, 1e-12)), 0))
    return np.sqrt((kl(p, m) + kl(q, m)) / 2)

def euclid_pair_ratio(X, gi, gj):
    """Pairwise-ratio analogue with Euclidean distances, for low-level feature spaces."""
    def md(P, Q, same):
        d = np.sqrt(((P[:, None] - Q[None]) ** 2).sum(-1))
        return d[~np.eye(len(P), dtype=bool)].mean() if same else d.mean()
    A, B = X[gi], X[gj]
    return md(A, B, False) / ((md(A, A, True) + md(B, B, True)) / 2)

# style-pair level low-level distances
LOW = {name: np.array([abs(style_mean(Z[:, q], i) - style_mean(Z[:, q], j)) for i, j in IU])
       for q, name in enumerate(STAT_NAMES)}
LOW["colour histogram (JS)"] = np.array([js_distance(style_mean(HIST, i), style_mean(HIST, j)) for i, j in IU])
LOWLEVEL_FEATS = np.column_stack([Z, SQH * np.sqrt(Z.shape[1])])   # scalars and histogram, comparable scale
LOW["combined low-level ratio"] = np.log([euclid_pair_ratio(LOWLEVEL_FEATS, np.where(labels == i)[0],
                                                             np.where(labels == j)[0]) for i, j in IU])

print("Are crossing pairs simply more different in low-level terms? (mean over pairs)\n")
print(pd.DataFrame({k: [v[CROSS].mean(), v[~CROSS].mean()] for k, v in LOW.items()},
                   index=["crossing pairs", "other pairs"]).T.round(3).to_string())

def crossing_perm_p(y, extra, sign=None):
    """Exact one-sided p for the crossing coefficient, in the observed direction:
    refit with each of the 420 groupings' crossing masks."""
    coef = lambda mask: ols(y, np.column_stack([mask.astype(float)] + extra))[1]
    obs = coef(CROSS); null = np.array([coef(m) for m in GROUPING_MASKS])
    sign = np.sign(obs) if obs != 0 else 1.0
    return obs, float(((sign * null) >= sign * obs - 1e-12).mean())

rows = []
for c in CONDS_S:
    y = PAIR_ALL[c]; start = np.log(r0s[c[0]])
    sign = None                                   # p is taken in the observed direction
    b, p = crossing_perm_p(y, [start], sign)
    rows.append(dict(condition=NAME_S[c], control="starting distance only", crossing_coef=b, exact_p=p))
    for name, v in LOW.items():
        b, p = crossing_perm_p(y, [start, v], sign)
        r_low = np.corrcoef(v, y)[0, 1]
        rows.append(dict(condition=NAME_S[c], control=f"+ {name}", crossing_coef=b, exact_p=p, corr_change_with_stat=r_low))
    b, p = crossing_perm_p(y, [start] + [LOW[n] for n in STAT_NAMES] + [LOW["colour histogram (JS)"]], sign)
    rows.append(dict(condition=NAME_S[c], control="+ all statistics together (overfits: 28 pairs)", crossing_coef=b, exact_p=p))
STYLEPAIR_REG = pd.DataFrame(rows)
print("\nCrossing coefficient across the 28 style pairs (exact one-sided p in the observed direction, from the 420 groupings; negative = crossing pairs converge more):\n")
print(STYLEPAIR_REG.set_index(["condition", "control"]).round(4).to_string())
for c in CONDS_S:
    t = STYLEPAIR_REG[(STYLEPAIR_REG.condition == NAME_S[c])]
    SUMMARY_C[f"image stats, style pairs · {NAME_S[c]}"] = {r.control: dict(coef=round(float(r.crossing_coef), 4), p=round(float(r.exact_p), 4))
                                                       for r in t.itertuples()}

In [ ]:
# painting-pair level: every pair among the 720 abstract + representational paintings
AR = np.concatenate([GA, GB])
iu = np.triu_indices(len(AR), 1)
lab_ar = labels[AR]
is_abs = np.isin(lab_ar, [class_names.index(n) for n in ABSTRACT])
crossing_pp = (is_abs[iu[0]] != is_abs[iu[1]]).astype(float)
same_style_pp = (lab_ar[iu[0]] == lab_ar[iu[1]]).astype(float)
def pdist_sq(X):
    sq = (X ** 2).sum(1); return np.maximum(sq[:, None] + sq[None] - 2 * X @ X.T, 0)
scalar_dist = np.sqrt(pdist_sq(Z[AR]))[iu]
hist_dist = np.sqrt(pdist_sq(SQH[AR]) / 2)[iu]                     # Hellinger distance
std = lambda v: (v - v.mean()) / v.std()

rows = []
for c in CONDS_S:
    S0 = unit_rows(E0[c[0]][AR]); s0 = (S0 @ S0.T)[iu]
    dsim = np.mean([((lambda E: E @ E.T)(unit_rows(E1[(c[0], c[1], s)][AR])))[iu] - s0 for s in SEEDS_OF_S[c]], 0)
    y = dsim / dsim.std()
    base = [crossing_pp, same_style_pp, std(s0)]
    b_base = ols(y, np.column_stack(base))
    b_low = ols(y, np.column_stack(base + [std(scalar_dist), std(hist_dist)]))
    rows.append(dict(condition=NAME_S[c], crossing_coef_without_lowlevel=b_base[1], crossing_coef_with_lowlevel=b_low[1],
                     share_kept=b_low[1] / b_base[1] if b_base[1] != 0 else np.nan,
                     scalar_stats_coef=b_low[4], colour_hist_coef=b_low[5],
                     corr_change_with_scalar_dist=np.corrcoef(dsim, scalar_dist)[0, 1],
                     corr_change_with_hist_dist=np.corrcoef(dsim, hist_dist)[0, 1]))
    SUMMARY_C[f"image stats, painting pairs · {NAME_S[c]}"] = {k: round(float(v), 4) for k, v in rows[-1].items() if k != "condition"}
print(f"{len(y):,} painting pairs. Outcome: change in cosine similarity (seed mean), in standard-deviation units.")
print("Crossing coefficient > 0 = crossing pairs gained similarity beyond other different-style pairs.\n")
print(pd.DataFrame(rows).set_index("condition").round(4).T.to_string())

# Part D: style probes — accessibility vs presence

A simple probe (linear, k-NN) measures how *readily* style can be read out (accessibility); the strongest probe we can fit estimates whether the information is *there at all* (presence; Pimentel et al., 2020). k-NN is reported alongside linear as in DINO and DINOv2.

| Part | What it does | Answers |
|---|---|---|
| D1 | Linear (as in Part B), linear with tuned C, k-NN (plain and DINO-weighted), tuned MLP; **presence = the probe with the best cross-validated score on the training paintings** | accessibility vs presence after each kind of training |
| D2 | Wilson 95% interval for every accuracy; for every before→after change a **paired** bootstrap over the 192 test paintings and an exact McNemar test per seed | which changes exceed test-set noise (all seeds share the same 192 test paintings, so the seed spread hides it) |
| D3 | Few-shot probes fitted **only on the 154 validation paintings** (never seen in fine-tuning), compared with the same probe fitted on fine-tuning paintings | is label training's few-shot advantage accessible style, or memorisation of trained paintings? |

In [ ]:
import warnings
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SHOTS = [5, 10]               # few-shot sizes per style; "all" (≈19 validation paintings per style) is added
N_DRAWS = 20
N_BOOT = 5000
MLP_GRID = {"mlpclassifier__hidden_layer_sizes": [(256,), (512,)],
            "mlpclassifier__alpha": [1e-3, 1e-2, 1e-1, 1.0, 10.0]}
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
MLP_MAX_ITER = 5000
KEYS = [(m, "before", None) for m in MODELS_S] + [(c[0], c[1], s) for c in CONDS_S for s in SEEDS_OF_S[c]]
emb_of = lambda k: E0[k[0]] if k[1] == "before" else E1[k]
cond_name = lambda m, o: f"{m} · {o}"
print(f"fine-tuning {len(fit_idx)} / validation {len(val_idx)} / test {len(te_idx)}")

## D0. Probes

In [ ]:
def unit(E):
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def knn_predict(E, tr, te, k=10, weighted=False, T=0.07):
    U = unit(E.astype(np.float64)); S = U[te] @ U[tr].T
    k = min(k, len(tr))
    nn = np.argpartition(-S, k - 1, axis=1)[:, :k]
    w = np.exp(np.take_along_axis(S, nn, 1) / T) if weighted else np.ones(nn.shape)
    votes = np.stack([(w * (labels[tr][nn] == c)).sum(1) for c in range(K)], 1)
    return votes.argmax(1)

def linear_predict(E, tr, te, C=1.0):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    return pipe.fit(E[tr], labels[tr]).predict(E[te])

def cv_acc(predict, E, tr, folds=5):
    acc = []
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=0).split(tr, labels[tr]):
        acc.append((predict(E, tr[a], tr[b]) == labels[tr[b]]).mean())
    return float(np.mean(acc))

def run_probes(E):
    """Returns {probe: (test predictions, cross-validated accuracy on the training paintings)}."""
    out = {}
    out["linear"] = (linear_predict(E, tr_idx, te_idx), cv_acc(linear_predict, E, tr_idx))
    cv_c = {C: cv_acc(lambda E_, a, b: linear_predict(E_, a, b, C=C), E, tr_idx) for C in C_GRID}
    best_c = max(cv_c, key=cv_c.get)
    out["linear (tuned C)"] = (linear_predict(E, tr_idx, te_idx, C=best_c), cv_c[best_c])
    out["k-NN (k=10)"] = (knn_predict(E, tr_idx, te_idx), cv_acc(knn_predict, E, tr_idx))
    wk = lambda E_, a, b: knn_predict(E_, a, b, k=20, weighted=True)
    out["k-NN weighted (k=20, DINO)"] = (wk(E, tr_idx, te_idx), cv_acc(wk, E, tr_idx))
    gs = GridSearchCV(make_pipeline(StandardScaler(), MLPClassifier(solver="lbfgs", max_iter=MLP_MAX_ITER, random_state=0)),
                      MLP_GRID, cv=StratifiedKFold(5, shuffle=True, random_state=0), n_jobs=-1)   # same 5 folds as the other probes
    gs.fit(E[tr_idx], labels[tr_idx])
    out["MLP (tuned)"] = (gs.predict(E[te_idx]), float(gs.best_score_))
    n_iter = int(gs.best_estimator_[-1].n_iter_)
    return out, {"C": best_c, "MLP": gs.best_params_, "MLP iterations": n_iter, "MLP converged": n_iter < MLP_MAX_ITER}

PRED, CVS, SETTINGS = {}, {}, {}
for k in KEYS:
    res, SETTINGS[k] = run_probes(emb_of(k))
    for p, (pred, cv) in res.items():
        PRED[(k, p)] = pred; CVS[(k, p)] = cv
    best = max(res, key=lambda p: res[p][1])
    PRED[(k, "presence (best by CV)")] = res[best][0]; CVS[(k, "presence (best by CV)")] = res[best][1]
    SETTINGS[k]["presence probe"] = best
    print(f"{k[0]} · {k[1]}{'' if k[2] is None else f' · seed {k[2]}'}: presence probe = {best}; tuned {SETTINGS[k]}")
unconverged = [k for k in KEYS if not SETTINGS[k]["MLP converged"]]
print("\nMLP converged for every embedding." if not unconverged else f"\nWARNING: MLP hit the iteration limit for {unconverged}; raise MLP_MAX_ITER.")
PROBE_NAMES = ["linear", "linear (tuned C)", "k-NN (k=10)", "k-NN weighted (k=20, DINO)", "MLP (tuned)", "presence (best by CV)"]

## D1–D2. Accuracy, 95% intervals and paired before→after tests

- **Accuracy**: mean over seeds, with a Wilson 95% interval for 192 test paintings (the test-set sampling error that the seed spread does not show).
- **Change**: per test painting, correctness after training (averaged over seeds) minus correctness before; 95% interval from a paired bootstrap over paintings.
- **McNemar p**: exact test on the paintings that flip between correct and wrong, one value per seed.

**Caveat:** these probes are fitted on the 768 non-test paintings, which include the 614 fine-tuning paintings. Test accuracy is still measured on unseen paintings, but for label-trained models the probe is fitted on paintings whose style labels the model was trained on. Section D3 gives the leakage-free version.

In [ ]:
def wilson(p, n=len(te_idx), z=1.96):
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h

rng = np.random.default_rng(0)
BOOT = rng.integers(0, len(te_idx), size=(N_BOOT, len(te_idx)))
y_te = labels[te_idx]

def mcnemar_p(c0, c1):
    b, c = int(((c0 == 1) & (c1 == 0)).sum()), int(((c0 == 0) & (c1 == 1)).sum())
    return 1.0 if b + c == 0 else binomtest(c, b + c, 0.5).pvalue

rows, SUMMARY_D = [], {"probes": {}, "settings": {f"{k[0]} · {k[1]} · {k[2]}": {kk: str(v) for kk, v in s.items()} for k, s in SETTINGS.items()}}
for m in MODELS_S:
    for o in ["before"] + [c[1] for c in CONDS_S if c[0] == m]:
        keys = [k for k in KEYS if k[0] == m and k[1] == o]
        for p in PROBE_NAMES:
            corr = np.array([(PRED[(k, p)] == y_te).astype(float) for k in keys])        # seeds × paintings
            acc = corr.mean()
            lo, hi = wilson(acc)
            row = dict(condition=cond_name(m, o), probe=p, accuracy=acc, ci95=f"[{lo:.3f}, {hi:.3f}]",
                       seed_sd=corr.mean(1).std(ddof=1) if len(keys) > 1 else 0.0)
            if o != "before":
                c0 = (PRED[((m, "before", None), p)] == y_te).astype(float)
                d = corr.mean(0) - c0
                boots = d[BOOT].mean(1)
                bl, bh = np.quantile(boots, [0.025, 0.975])
                ps = [mcnemar_p(c0, c) for c in corr]
                row.update(change=d.mean(), change_ci95=f"[{bl:+.3f}, {bh:+.3f}]",
                           excludes_zero=bool(bl > 0 or bh < 0), mcnemar_p_per_seed=", ".join(f"{x:.3f}" for x in ps))
                SUMMARY_D["probes"][f"{cond_name(m, o)} | {p}"] = dict(acc=round(float(acc), 4), change=round(float(d.mean()), 4),
                                                                     ci=[round(float(bl), 4), round(float(bh), 4)],
                                                                     mcnemar=[round(float(x), 4) for x in ps])
            else:
                SUMMARY_D["probes"][f"{cond_name(m, o)} | {p}"] = dict(acc=round(float(acc), 4))
            rows.append(row)
TAB = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 40)
print("Test accuracy (192 paintings; chance = 0.125). Probes fitted on the 768 non-test paintings,")
print("which include the 614 fine-tuning paintings; see D3 for probes fitted only on unseen paintings.\n")
print(TAB.pivot(index="condition", columns="probe", values="accuracy")[PROBE_NAMES].round(3).to_string())
print("\nChange from the untrained model, paired 95% interval, McNemar p per seed:\n")
print(TAB[TAB.change.notna()][["condition", "probe", "change", "change_ci95", "excludes_zero", "mcnemar_p_per_seed"]]
      .set_index(["condition", "probe"]).round(3).to_string())

## D3. Few-shot probes without leakage

Each probe is fitted on *n* paintings per style drawn from the **validation paintings** (never seen in fine-tuning) and tested on the 192 test paintings. For comparison, the same probe is also fitted on *n* paintings per style from the **fine-tuning paintings**: if label training's advantage exists only there, it was memorisation of trained paintings rather than accessible style.

The gap to the untrained model gets a paired 95% bootstrap interval (correctness averaged over draws and seeds per test painting).

In [ ]:
n_all = int(np.bincount(labels[val_idx]).min())
sizes = SHOTS + [n_all]

def draw(pool, n, rng):
    return np.concatenate([rng.choice(pool[labels[pool] == c], n, replace=False) for c in range(K)])

few_probes = {"linear": lambda E, tr: linear_predict(E, tr, te_idx),
              "k-NN (k=5)": lambda E, tr: knn_predict(E, tr, te_idx, k=5)}
FEW_CORR = {}
for k in KEYS:
    E = emb_of(k)
    for pool_name, pool in (("validation (unseen)", val_idx), ("fine-tuning (seen)", fit_idx)):
        for n in sizes:
            for p, f in few_probes.items():
                r = np.random.default_rng(1000 * n + (0 if k[2] is None else k[2]))
                draws = N_DRAWS
                FEW_CORR[(k, pool_name, n, p)] = np.mean([(f(E, draw(pool, n, r)) == y_te) for _ in range(draws)], 0)

rows = []
for m in MODELS_S:
    for o in ["before"] + [c[1] for c in CONDS_S if c[0] == m]:
        keys = [k for k in KEYS if k[0] == m and k[1] == o]
        for pool_name in ("validation (unseen)", "fine-tuning (seen)"):
            for n in sizes:
                for p in few_probes:
                    c1 = np.mean([FEW_CORR[(k, pool_name, n, p)] for k in keys], 0)
                    row = dict(condition=cond_name(m, o), fitted_on=pool_name, per_style=n, probe=p, accuracy=c1.mean())
                    if o != "before":
                        c0 = FEW_CORR[((m, "before", None), pool_name, n, p)]
                        d = c1 - c0; boots = d[BOOT].mean(1)
                        bl, bh = np.quantile(boots, [0.025, 0.975])
                        row.update(gain=d.mean(), gain_ci95=f"[{bl:+.3f}, {bh:+.3f}]", excludes_zero=bool(bl > 0 or bh < 0))
                        SUMMARY_D.setdefault("few-shot", {})[f"{cond_name(m, o)} | {pool_name} | {n}/style | {p}"] = dict(
                            acc=round(float(c1.mean()), 4), gain=round(float(d.mean()), 4), ci=[round(float(bl), 4), round(float(bh), 4)])
                    rows.append(row)
FEW = pd.DataFrame(rows)
print(f"Few-shot test accuracy ('all' = {n_all} per style):\n")
print(FEW.pivot_table(index=["probe", "fitted_on", "condition"], columns="per_style", values="accuracy").round(3).to_string())
print("\nGain over the untrained model, probe fitted on VALIDATION paintings only (the leakage-free version):\n")
print(FEW[(FEW.fitted_on == "validation (unseen)") & FEW.gain.notna()]
      .set_index(["probe", "condition", "per_style"])[["gain", "gain_ci95", "excludes_zero"]].round(3).to_string())

fig, axes = plt.subplots(len(few_probes), len(MODELS_S), figsize=(6 * len(MODELS_S), 3.6 * len(few_probes)), squeeze=False)
for r_, p in enumerate(few_probes):
    for col, m in enumerate(MODELS_S):
        ax = axes[r_][col]
        for o, st in (("before", "k:"), ("labels", "-"), ("exposure", "--")):
            for pool_name, mk in (("validation (unseen)", "o"), ("fine-tuning (seen)", "x")):
                t = FEW[(FEW.condition == cond_name(m, o)) & (FEW.probe == p) & (FEW.fitted_on == pool_name)].sort_values("per_style")
                if len(t):
                    ax.plot(t.per_style, t.accuracy, st, marker=mk, label=f"{o}, fit on {pool_name.split()[0]}")
        ax.set_title(f"{m} · {p}", fontsize=10); ax.set_xlabel("training paintings per style"); ax.set_ylabel("test accuracy")
        ax.set_xticks(sizes); ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Summary of Parts C and D
Prints every number from Parts C and D in one block (handy to paste or save).

In [ ]:
print(json.dumps({"Part C": SUMMARY_C, "Part D": SUMMARY_D}, indent=1, ensure_ascii=False, default=str))